In [ ]:
from import_ipynb import NotebookFinder  # type: ignore
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score, roc_auc_score
import importlib
import os
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
import numpy as np


In [ ]:
# retrouver les dossiers
root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
utils_dir = os.path.join(root, "pneumonia_knn", "notebooks", "utils")

# --- pneumonia_knn\notebooks\utils
spec_print = NotebookFinder().find_spec("prints", [utils_dir])
prints = importlib.util.module_from_spec(spec_print)
spec_print.loader.exec_module(prints)

In [ ]:
# 1. Définir les métriques pour GridSearchCV
scoring = {
    'accuracy':  'accuracy',
    'precision': make_scorer(precision_score, average='weighted'),
    'recall':    make_scorer(recall_score,    average='macro'),
    'f1':        make_scorer(f1_score,        average='weighted'),
    'roc_auc':   make_scorer(roc_auc_score,   response_method="predict_proba", average='macro', multi_class='ovo')
}
def get_scoring():
    return scoring
# 2. Colonnes à afficher dans les résultats
cols = [
    'params',
    'mean_test_accuracy',
    'mean_test_precision',
    'mean_test_recall',
    'mean_test_f1',
    'mean_test_roc_auc',
    'rank_test_accuracy'
]

### Visualiser les composantes principales du PCA
Cette fonction affiche les premières composantes principales sous forme d'images.

In [ ]:
def plot_pca_components(pca):
    fig, axes = plt.subplots(1, 5, figsize=(14, 3))

    for i, ax in enumerate(axes):
        composante = pca.components_[i].reshape(224, 224, 3)  # ← 3 canaux
        composante = (composante - composante.min()) / (composante.max() - composante.min())
        
        ax.imshow(composante)  # ← pas cmap='gray'
        ax.set_title(f'PC{i+1} — {pca.explained_variance_ratio_[i]*100:.1f}%', fontsize=9)
        ax.axis('off')

    plt.suptitle("Ce que chaque composante PCA a capturé", fontsize=11)
    plt.tight_layout()
    plt.show()

### Visualiser la variance expliquée par le PCA
Ce graphique montre la variance expliquée et la variance cumulée par composante principale.

In [ ]:
def plot_pca_variance_explained(pca):
    variance_expliquee = pca.explained_variance_ratio_ * 100
    variance_cumulee = np.cumsum(variance_expliquee)
    fig, ax1 = plt.subplots(figsize=(10, 6))

    # Barres - variance par composante
    ax1.bar(range(1, len(variance_expliquee) + 1), variance_expliquee, color='steelblue', alpha=0.7, label='Variance expliquée par composante')
    ax1.set_xlabel('Composantes principales')
    ax1.set_ylabel('Variance expliquée (%)', color='steelblue')

    # Courbe - variance cumulée
    ax2 = ax1.twinx()
    ax2.plot(range(1, len(variance_cumulee) + 1), variance_cumulee, color='red', marker='o', markersize=0.5, label='Variance cumulée')
    ax2.set_ylabel('Variance cumulée (%)', color='red')
    ax2.axhline(y=95, color='gray', linestyle='--', label='Seuil 95%')

    plt.title('Scree Plot — Variance expliquée par le PCA')
    fig.legend(loc='center right')
    plt.tight_layout()
    plt.show()

### Visualiser les courbes de performances GridSearchCV
Affiche l'évolution de l'accuracy, de la précision, du rappel et du F1 en fonction du nombre de voisins.

In [ ]:
def plot_grid_search_metric_curves(grid_search_pca):
    results_df = pd.DataFrame(grid_search_pca.cv_results_)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Résultats GridSearchCV - KNN + PCA', fontsize=16, fontweight='bold')

    axes[0, 0].plot(results_df['param_n_neighbors'], results_df['mean_test_accuracy'], marker='o', label='Test Accuracy')
    axes[0, 0].set_xlabel('Nombre de voisins (n_neighbors)')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].set_title('Accuracy vs n_neighbors')
    axes[0, 0].grid(True)
    axes[0, 0].legend()

    axes[0, 1].plot(results_df['param_n_neighbors'], results_df['mean_test_precision'], marker='s', label='Test Precision', color='orange')
    axes[0, 1].set_xlabel('Nombre de voisins (n_neighbors)')
    axes[0, 1].set_ylabel('Precision')
    axes[0, 1].set_title('Precision vs n_neighbors')
    axes[0, 1].grid(True)
    axes[0, 1].legend()

    axes[1, 0].plot(results_df['param_n_neighbors'], results_df['mean_test_recall'], marker='^', label='Test Recall', color='green')
    axes[1, 0].set_xlabel('Nombre de voisins (n_neighbors)')
    axes[1, 0].set_ylabel('Recall')
    axes[1, 0].set_title('Recall vs n_neighbors')
    axes[1, 0].grid(True)
    axes[1, 0].legend()

    axes[1, 1].plot(results_df['param_n_neighbors'], results_df['mean_test_f1'], marker='d', label='Test F1-Score', color='red')
    axes[1, 1].set_xlabel('Nombre de voisins (n_neighbors)')
    axes[1, 1].set_ylabel('F1-Score')
    axes[1, 1].set_title('F1-Score vs n_neighbors')
    axes[1, 1].grid(True)
    axes[1, 1].legend()

    plt.tight_layout()
    plt.savefig('../documents/model/hyperparameter/knn_pca_grid_search_results.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("✅ Image sauvegardée : ../documents/model/hyperparameter/knn_pca_grid_search_results.png")

### Afficher un aperçu des métriques GridSearchCV
Crée une image montrant les colonnes disponibles dans les résultats GridSearchCV.

In [ ]:
def plot_grid_search_metric_overview_table(results, title):
    # Sélectionner les colonnes clés
    table_df = results[[
        'params',
        'mean_test_accuracy',
        'mean_test_precision',
        'mean_test_recall',
        'mean_test_f1',
        'mean_test_roc_auc',
        'rank_test_accuracy'
    ]].copy()

    # Convertir les params en chaîne, puis tronquer pour l'affichage
    table_df['params'] = table_df['params'].apply(lambda p: str(p))
    table_df['params'] = table_df['params'].apply(
        lambda x: x if len(x) <= 120 else x[:117] + '...'
    )

    fig, ax = plt.subplots(figsize=(16, 2 + 0.45 * len(table_df)))
    ax.axis('off')

    table = ax.table(
        cellText=table_df.values,
        colLabels=table_df.columns,
        cellLoc='center',
        loc='center',
        colColours=['#f7f7f7'] + ['#ffffff'] * (len(table_df.columns) - 1)
    )

    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.5)

    # Styliser l'en-tête
    for j in range(len(table_df.columns)):
        table[(0, j)].set_facecolor('#40466e')
        table[(0, j)].set_text_props(weight='bold', color='white')

    # Alternance de couleur sur les lignes
    for i in range(1, len(table_df) + 1):
        for j in range(len(table_df.columns)):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#f0f0f0')

    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    output_path = '../documents/model/hyperparameter/grid_search_metrics_table.png'
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✅ Tableau sauvegardé : {output_path}")

### Afficher le tableau GridSearchCV complet
Affiche les colonnes d'intérêt triées par rang et sauvegarde une image du tableau.

In [ ]:
def display_grid_search_results_table(results, title):
    print(title)
    display(results[cols].sort_values('rank_test_accuracy'))

### Afficher les résultats triés de GridSearchCV
Présente un tableau des meilleures configurations selon le rang d'accuracy.

In [ ]:
def plot_knn_prediction_scatter(y_test, y_pred, X_reduced, implementation_name):
    class_names = ['NORMAL', 'BACTERIA', 'VIRUS']
    colors      = ['#378ADD', '#22a05a', '#EF9F27']
    
    correct = (y_test == y_pred)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Projection {implementation_name} — KNN predictions", fontsize=14)
    
    # --- Onglet 1 : Vraie classe ---
    ax = axes[0]
    for c, (name, color) in enumerate(zip(class_names, colors)):
        mask = y_test == c
        ax.scatter(X_reduced[mask, 0], X_reduced[mask, 1],
                   c=color, label=name, alpha=0.6, s=15)
    ax.set_title("Vraie classe")
    ax.set_xlabel(f"{implementation_name}1")
    ax.set_ylabel(f"{implementation_name}2")
    ax.legend(fontsize=8)
    
    # --- Onglet 2 : Prédiction KNN ---
    ax = axes[1]
    for c, (name, color) in enumerate(zip(class_names, colors)):
        mask = y_pred == c
        ax.scatter(X_reduced[mask, 0], X_reduced[mask, 1],
                   c=color, label=f"Prédit {name}", alpha=0.6, s=15)
    ax.set_title("Prédiction KNN")
    ax.set_xlabel(f"{implementation_name}1")
    ax.set_ylabel(f"{implementation_name}2")
    ax.legend(fontsize=8)
    
    # --- Onglet 3 : Erreurs ---
    ax = axes[2]
    ax.scatter(X_reduced[correct, 0],  X_reduced[correct, 1],
               c='#cccccc', alpha=0.3, s=12, label='Correct')
    ax.scatter(X_reduced[~correct, 0], X_reduced[~correct, 1],
               c='#e24b4a', alpha=0.8, s=20, label='Erreur')
    ax.set_title("Erreurs de prédiction")
    ax.set_xlabel(f"{implementation_name}1")
    ax.set_ylabel(f"{implementation_name}2")
    ax.legend(fontsize=8)
    
    plt.tight_layout()
    plt.show()

### Visualiser les prédictions KNN
Affiche la vraie classe, les prédictions et les erreurs dans des nuages de points.

In [ ]:
def plot_prediction_metrics_bar(y_test, y_pred, title = "Métriques de prédiction", y_proba = None):
    metrics = {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='weighted'),
        'Recall':    recall_score(y_test, y_pred, average='weighted'),
        'F1':        f1_score(y_test, y_pred, average='weighted'),
    }
    
    if y_proba is not None:
        metrics['ROC AUC'] = roc_auc_score(y_test, y_proba, average='macro', multi_class='ovo')

    fig, ax = plt.subplots(figsize=(14, 6))  # ← plus large et plus haut
    
    bars = ax.barh(
        list(metrics.keys()), 
        list(metrics.values()), 
        color=['#378ADD', '#22a05a', '#EF9F27', '#e24b4a', '#9b59b6'],  # une couleur par barre
        alpha=0.8
    )    
    for bar, val in zip(bars, metrics.values()):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=8)  # ← police plus petite
    
    ax.set_xlim(0, 1.15)
    ax.set_xlabel("Score", fontsize=9)        # ← plus petit
    ax.set_title(title, fontsize=11)          # ← plus petit
    ax.tick_params(axis='both', labelsize=8)  # ← étiquettes axes plus petites
    ax.axvline(x=1.0, color='gray', linestyle='--', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

### Visualiser les scores de prédiction
Affiche un histogramme des métriques globales du modèle KNN.

In [ ]:
def plot_true_positives_by_class(y_test, y_pred, title=""):
    classes = ['NORMAL', 'BACTERIA', 'VIRUS']
    cm = confusion_matrix(y_test, y_pred)
    
    # Les vrais positifs sont sur la diagonale
    tp = cm.diagonal()
    total = cm.sum(axis=1)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    
    x = np.arange(len(classes))
    bars_tp    = ax.bar(x - 0.2, tp,    width=0.4, label='Vrais positifs', color='#22a05a', alpha=0.8)
    bars_total = ax.bar(x + 0.2, total, width=0.4, label='Total réel',     color='#378ADD', alpha=0.8)
    
    # Afficher les valeurs sur les barres
    for bar in bars_tp:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(int(bar.get_height())), ha='center', fontsize=9)
    for bar in bars_total:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(int(bar.get_height())), ha='center', fontsize=9)
    
    ax.set_xticks(x)
    ax.set_xticklabels(classes, fontsize=10)
    ax.set_ylabel("Nombre d'images", fontsize=9)
    ax.set_title(f"Vrais positifs par classe — {title}", fontsize=11)
    ax.legend(fontsize=9)
    
    plt.tight_layout()
    plt.show()


### Visualiser les vrais positifs par classe
Compare les vrais positifs et les totaux réels pour chaque classe.